# Ship Detection from Drive

## 1. Config

In [ ]:
import os

CFG = dict(
    MODEL_PATH = "/kaggle/input/models/hunglq/pretrained/pytorch/default/1/yolo11s_tci.pt",
    HF_FALLBACK_REPO = "mayrajeo/marine-vessel-yolo",
    HF_FALLBACK_FILE = "yolo11s_tci.pt",
    DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/12QVIJ3H4_h_YeKHEJ8U2387wuMKRd4Qz",  # <<< SỬA
    LIMIT = None,
    TILE = 320,
    OVERLAP = 64,
    IMGSZ = 320,
    CONF = 0.1,
    IOU_NMS = 0.7,
    NMS_IOU = 0.5,
    BATCH = 16,
    SKIP_DARK = 5,
    DEVICE = "auto",

    # --- Cloud masking (Sentinel-2 L1C/L2A) ---
    # Vì sao: mô hình hay nhận nhầm CỤM MÂY NHỎ, sáng thành tàu. Ta dùng
    # mask mây của Copernicus (SCL/MSK_CLDPRB/MSK_CLASSI/MSK_CLOUDS) để loại
    # các dự đoán rơi vào vùng mây. Nếu chỉ có ảnh TCI (không có metadata) thì
    # tự suy ra mask theo độ sáng/độ bão hoà (kém chính xác hơn).
    CLOUD_MASK = True,             # bật/tắt toàn bộ tính năng lọc mây
    CLOUD_MASK_MODE = "after",     # "after" (KHUYẾN NGHỊ: infer xong mới lọc) | "before" (xoá pixel mây trước khi infer) | "off"
    CLOUD_SOURCE = "auto",         # nguồn mask: "auto" (SCL/MSK_CLASSI/độ sáng) | "omnicloudmask" (CNN, chính xác hơn, cần band + nên có GPU)
    CLOUD_OCM_CLASSES = (1, 2),    # OmniCloudMask: 1=mây dày, 2=mây mỏng (thêm bóng mây=3 qua CLOUD_INCLUDE_SHADOW)
    CLOUD_OCM_RES = 20.0,          # GSD (m) khi chạy OCM: 10-50; 20 nhanh & đủ cho lọc box
    CLOUD_OCM_BATCH = 1,           # số patch mỗi batch (tăng nếu GPU khoẻ)
    CLOUD_OCM_DEVICE = None,       # None = tự chọn "cuda"/"cpu"; hoặc ép "cuda"/"cpu"
    CLOUD_FILTER = "fraction",     # cách quyết định 1 box là mây: "fraction" (theo % diện tích) | "center" (theo pixel tâm)
    CLOUD_FRAC_THR = 0.35,         # loại box nếu >= 35% diện tích của nó là mây (nâng lên ~0.5 nếu bị mask nhầm nhiều)
    CLOUD_PROTECT_CONF = 0.5,      # KHÔNG loại dự đoán có conf >= ngưỡng này Ở BƯỚC OCM (bảo vệ tàu thật; 0 = tắt bảo vệ)
    # --- Lọc cirrus bằng band B10 (chỉ L1C) — bắt cả FP mây conf cao mà PROTECT_CONF che ---
    CLOUD_B10 = True,              # (L1C) bỏ box SÁNG ở B10 1375nm (mây/cirrus); tàu tối ở B10 -> giữ. L2A tự bỏ qua.
    CLOUD_B10_THR = 0.010,         # ngưỡng chênh reflectance B10 so với nền để coi là mây (giảm = bỏ nhiều hơn)
    CLOUD_B10_BG_PCT = 20,         # phân vị B10 lấy làm nền (nước/tối)
    CLOUD_SCL_CLASSES = (8, 9, 10),# L2A SCL: 8=mây vừa, 9=mây cao, 10=cirrus
    CLOUD_INCLUDE_SHADOW = False,  # thêm SCL 3 (bóng mây) vào mask
    CLOUD_INCLUDE_SNOW = False,    # thêm SCL 11 (tuyết/băng) vào mask
    CLOUD_CLDPRB_THR = 40,         # ngưỡng % cho MSK_CLDPRB (L2A)
    CLOUD_DILATE_PX = 0,           # nới rộng mask thêm N pixel (theo lưới mask); 0 = không nới (tránh nuốt tàu gần mây)
    CLOUD_SKIP_IF_TCI_ONLY = True, # chỉ có ảnh TCI (không có band/mask kèm theo) -> BỎ QUA lọc mây cho ảnh đó (False = dùng fallback độ sáng)
    CLOUD_BRIGHT_THR = 0.60,       # fallback khi chỉ có TCI (chỉ dùng nếu CLOUD_SKIP_IF_TCI_ONLY=False): ngưỡng độ sáng (0-1)
    CLOUD_SAT_THR = 0.20,          # fallback khi chỉ có TCI: ngưỡng độ bão hoà tối đa
    DRAW_CLOUD = True,             # tô nền vùng mây (xám) khi vẽ overview để kiểm tra

    SAVE_GEOTIFF = True,
    SAVE_JP2 = False,
    SAVE_VECTOR = True,
    SAVE_CROPS_ZIP = True,         # mỗi ảnh -> 1 file {stem}_crops.zip chứa toàn bộ crop của các phát hiện trong ảnh đó
    CROPS_PAD = 40,                # lề (px) quanh mỗi box khi cắt crop
    CROPS_DRAW_BOX = True,         # vẽ khung đỏ lên crop
    # --- Hard-negative mining: xuất tile mây thành mẫu NEGATIVE để fine-tune lại model ---
    SAVE_HARD_NEG = True,          # xuất tile hard-negative (ảnh + nhãn rỗng) -> gói hard_negatives.zip
    HARD_NEG_SOURCE = "dropped",   # "dropped" (mây đã bị lọc — tự động & AN TOÀN) | "kept" (các FP còn sót — hard nhất, NHỚ xoá tile con tàu thật trước khi train) | "all"
    HARD_NEG_TILE = 320,           # kích thước tile negative (nên khớp TILE lúc train)
    BURN_MAX_MP = 400,
    OVERVIEW_MAX = 1600,
    LINE_THICKNESS = 1,
    DRAW_GT = False,
    EXPORT_MODE = "both", # pred | gt | both
    GT_GPKG_PATH = "/kaggle/input/zenodo-vessel-gpkg",
    GT_LAYER = None,
    UNZIP_DELETE_ZIP = False,      # xoá file .zip sau khi giải nén (tiết kiệm dung lượng Kaggle)
    DL_DIR = "/kaggle/working/drive_tci",
    OUT_DIR = "/kaggle/working/drive_infer",
)
os.makedirs(CFG["DL_DIR"], exist_ok=True)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
print("Confidence threshold:", CFG["CONF"])
print("Cloud masking:", CFG["CLOUD_MASK"], "| mode:", CFG["CLOUD_MASK_MODE"])


## 2. Ultralytics for YOLO

In [ ]:
%pip install -q -U ultralytics "huggingface_hub>=0.24" rasterio geopandas gdown omnicloudmask

import ultralytics
ultralytics.checks()
print("ultralytics:", ultralytics.__version__)
import rasterio, geopandas
_jp2 = rasterio.drivers.raster_driver_extensions().get("jp2", "")

In [3]:
import torch

def pick_device(pref):
    if pref not in ("auto", None, ""):
        print("device preference:", pref)
        return str(pref)
    if not torch.cuda.is_available():
        print("cuda not available")
        return "cpu"
    try:
        _ = (torch.zeros(16, device="cuda") + 1).sum().item()
        torch.cuda.synchronize()
        print("gpu detected:", torch.cuda.get_device_name(0))
        return "0"
    except Exception as e:
        print("Not found GPU kernel:", str(e).splitlines()[0])
        return "cpu"

DEVICE_EFF = pick_device(CFG["DEVICE"])
print("device used =", DEVICE_EFF)

gpu detected: Tesla T4
device used = 0


## 3. Apply model

In [4]:
from ultralytics import YOLO

def resolve_model(cfg):
    p = cfg["MODEL_PATH"]
    if p and os.path.exists(p):
        print("Used model:", p)
        return p
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=cfg["HF_FALLBACK_REPO"], filename=cfg["HF_FALLBACK_FILE"],
                           local_dir="/kaggle/working/weights")

MODEL_WEIGHTS = resolve_model(CFG)
model = YOLO(MODEL_WEIGHTS)

Used model: /kaggle/input/models/hunglq/pretrained/pytorch/default/1/yolo11s_tci.pt


## 4. Inference, Output

In [ ]:
import os, json as _json
import numpy as np
import rasterio
from rasterio.windows import Window
from PIL import Image
import torch
from torchvision.ops import nms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import cv2
import geopandas as gpd
from shapely.geometry import box as shp_box


def _band_idx(nbands):
    return [1, 2, 3] if nbands >= 3 else [1, 1, 1]


def _to_uint8(arr):
    if arr.dtype == np.uint8:
        return arr
    a = arr.astype(np.float32)
    return np.clip(a / (float(a.max()) or 1.0) * 255.0, 0, 255).astype(np.uint8)


def _read_rgb_window(src, window):
    arr = src.read(indexes=_band_idx(src.count), window=window, boundless=True, fill_value=0)
    return _to_uint8(np.transpose(arr, (1, 2, 0)))


def _tile_offsets(size, tile, stride):
    if size <= tile:
        return [0]
    offs = list(range(0, size - tile + 1, stride))
    if offs[-1] != size - tile:
        offs.append(size - tile)
    return offs


def infer_large_raster(raster_path, model, conf=0.25, tile=320, overlap=64, imgsz=320, iou_nms=0.7, nms_iou=0.5, batch=16, skip_dark=5, device="cpu", verbose=True, cloud_mask=None):

    with rasterio.open(raster_path) as src:
        W, H, nb = src.width, src.height, src.count
        stride = max(1, tile - overlap)
        xs, ys = _tile_offsets(W, tile, stride), _tile_offsets(H, tile, stride)
        if verbose:
            print(f"Image info: {W}×{H}px, {nb} band -> {len(xs)}×{len(ys)} = {len(xs) * len(ys)} tile "
                  f"(tile={tile}, overlap={overlap}, imgsz={imgsz}).")

        boxes_all, scores_all = [], []
        buf_imgs, buf_off = [], []

        def _flush():
            if not buf_imgs:
                return
            for res, (ox, oy) in zip(
                    model.predict(buf_imgs, imgsz=imgsz, conf=conf, iou=iou_nms,
                                  device=device, verbose=False), buf_off):
                if res.boxes is None or len(res.boxes) == 0:
                    continue
                xyxy = res.boxes.xyxy.cpu().numpy().copy()
                xyxy[:, [0, 2]] += ox        # tile coordinate
                xyxy[:, [1, 3]] += oy
                boxes_all.append(xyxy)
                scores_all.append(res.boxes.conf.cpu().numpy())
            buf_imgs.clear(); buf_off.clear()

        n_used = 0
        for oy in ys:
            for ox in xs:
                arr = _read_rgb_window(src, Window(ox, oy, tile, tile))
                if cloud_mask is not None:
                    cmw = cloud_mask.window_mask(src.transform, ox, oy, tile, tile)
                    if cmw.any():
                        arr = arr.copy()
                        arr[cmw] = 0
                if int(arr.max()) < skip_dark:
                    continue
                buf_imgs.append(Image.fromarray(arr)); buf_off.append((ox, oy)); n_used += 1
                if len(buf_imgs) >= batch:
                    _flush()
        _flush()

    if verbose:
        print(f"Found {n_used} tile above dark threshold")
    if not boxes_all:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), (W, H)

    boxes = np.concatenate(boxes_all).astype(np.float32)
    scores = np.concatenate(scores_all).astype(np.float32)
    keep = nms(torch.from_numpy(boxes), torch.from_numpy(scores), nms_iou).numpy()
    return boxes[keep], scores[keep], (W, H)


def draw_overview(raster_path, boxes=None, gt_boxes=None, max_side=1600, title=None, save_path=None, show=True, show_pred=None, drop_boxes=None, cloud_mask=None):
    # draw ground truth
    show_pred = (boxes is not None) if show_pred is None else show_pred
    boxes = np.zeros((0, 4), np.float32) if boxes is None else np.asarray(boxes)
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        scale = min(1.0, max_side / max(W, H))
        ow, oh = max(1, int(W * scale)), max(1, int(H * scale))
        ov = _to_uint8(np.transpose(
            src.read(indexes=_band_idx(src.count), out_shape=(3, oh, ow)), (1, 2, 0)))
        _T = src.transform
    fig, ax = plt.subplots(figsize=(13, max(3, 13 * oh / ow)))
    ax.imshow(ov)
    if cloud_mask is not None:
        try:
            ovT = _T * rasterio.Affine.scale(W / ow, H / oh)
            cmov = cloud_mask.window_mask(ovT, 0, 0, ow, oh)
            overlay = np.zeros((oh, ow, 4), np.float32)
            overlay[cmov] = (0.55, 0.55, 0.55, 0.35)
            ax.imshow(overlay)
        except Exception:
            pass
    if drop_boxes is not None and len(drop_boxes):
        for (x1, y1, x2, y2) in (np.asarray(drop_boxes) * scale):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="orange", linewidth=0.8, linestyle="--"))
    if gt_boxes is not None and len(gt_boxes):
        for (x1, y1, x2, y2) in (np.asarray(gt_boxes) * scale):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="lime", linewidth=1.0))
    for (x1, y1, x2, y2) in (np.asarray(boxes) * scale):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor="red", linewidth=0.8))
    handles = []
    if show_pred:
        handles.append(Line2D([0], [0], color="red", lw=2, label="Dự đoán"))
    if drop_boxes is not None and len(drop_boxes):
        handles.append(Line2D([0], [0], color="orange", lw=2, linestyle="--", label="Bỏ (mây)"))
    if gt_boxes is not None and len(gt_boxes):
        handles.append(Line2D([0], [0], color="lime", lw=2, label="Ground truth"))
    if handles:
        ax.legend(handles=handles, loc="upper right", fontsize=9)
    ax.set_title(title or f"{os.path.basename(raster_path)} — {len(boxes)} dự đoán", fontsize=12)
    ax.axis("off")
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


def show_detection_crops(raster_path, boxes, scores, topk=12, pad=40, ncol=4, save_path=None, show=True):
    # show clear vessel
    if len(boxes) == 0:
        print("No vessel")
        return
    order = np.argsort(-scores)[:topk]
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        nrow = (len(order) + ncol - 1) // ncol
        fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow))
        axes = np.atleast_1d(axes).flatten()
        for ax, i in zip(axes, order):
            x1, y1, x2, y2 = boxes[i]
            cx0, cy0 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            cw = min(W - cx0, int(x2 - x1) + 2 * pad)
            ch = min(H - cy0, int(y2 - y1) + 2 * pad)
            ax.imshow(_read_rgb_window(src, Window(cx0, cy0, cw, ch)))
            ax.add_patch(patches.Rectangle((x1 - cx0, y1 - cy0), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="red", linewidth=1.0))
            ax.set_title(f"conf={scores[i]:.2f}", fontsize=9); ax.axis("off")
        for ax in axes[len(order):]:
            ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


def burn_boxes_to_raster(raster_path, out_path, pred_boxes, pred_scores=None, gt_boxes=None, driver="GTiff", thickness=None, jp2_quality=100, max_megapixels=400):

    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        mp = W * H / 1e6
        if mp > max_megapixels:
            raise MemoryError("Out of memory")
        img = np.ascontiguousarray(_read_rgb_window(src, Window(0, 0, W, H)))  # (H, W, 3) RGB uint8
        crs, transform = src.crs, src.transform

    t = max(1, int(thickness)) if thickness else 2

    def _draw(boxes, color):
        if boxes is None:
            return
        for b in np.asarray(boxes):
            x1, y1, x2, y2 = (int(round(v)) for v in b[:4])
            cv2.rectangle(img, (x1, y1), (x2, y2), color, t)

    _draw(gt_boxes, (0, 255, 0))
    _draw(pred_boxes, (255, 0, 0))

    profile = dict(driver=driver, width=W, height=H, count=3, dtype="uint8",
                   crs=crs, transform=transform)
    if driver == "GTiff":
        profile.update(compress="LZW", tiled=True, blockxsize=512, blockysize=512,
                       photometric="RGB", BIGTIFF="IF_SAFER")
    elif driver == "JP2OpenJPEG":
        profile.update(QUALITY=jp2_quality, REVERSIBLE=("YES" if jp2_quality >= 100 else "NO"))
    with rasterio.open(out_path, "w", **profile) as dst:
        for k in range(3):
            dst.write(img[:, :, k], k + 1)
    print(f"Written to: {out_path}")
    return out_path


def detections_to_vectors(raster_path, boxes, scores, out_geojson=None, out_gpkg=None):
    # Convert to georef, keep pixel info

    with rasterio.open(raster_path) as src:
        T, crs = src.transform, src.crs
    geoms, confs = [], []
    for (x1, y1, x2, y2), s in zip(boxes.tolist(), scores.tolist()):
        (X1, Y1) = rasterio.transform.xy(T, y1, x1)   # (row=y, col=x) -> (x_crs, y_crs)
        (X2, Y2) = rasterio.transform.xy(T, y2, x2)
        geoms.append(shp_box(min(X1, X2), min(Y1, Y2), max(X1, X2), max(Y1, Y2)))
        confs.append(round(float(s), 4))
    gdf = gpd.GeoDataFrame({"confidence": confs, "class": "vessel"}, geometry=geoms, crs=crs)
    if out_gpkg:
        gdf.to_file(out_gpkg, driver="GPKG")
        print("Vector gpkg", out_gpkg)
    if out_geojson:
        (gdf.to_crs(4326) if crs is not None else gdf).to_file(out_geojson, driver="GeoJSON")
        print("Geojson ", out_geojson)
    return gdf


def save_detections_json(raster_path, boxes, scores, out_json):
    crs, recs = None, []
    try:
        with rasterio.open(raster_path) as src:
            T, crs = src.transform, src.crs
            for (x1, y1, x2, y2), s in zip(boxes.tolist(), scores.tolist()):
                rec = {"confidence": round(float(s), 4),
                       "bbox_xyxy_pixel": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]}
                if crs is not None:
                    gx, gy = rasterio.transform.xy(T, (y1 + y2) / 2.0, (x1 + x2) / 2.0)
                    rec["center_crs_xy"] = [round(gx, 2), round(gy, 2)]
                recs.append(rec)
    except Exception as e:
        recs = [{"confidence": round(float(s), 4),
                 "bbox_xyxy_pixel": [round(float(v), 1) for v in b]}
                for b, s in zip(boxes.tolist(), scores.tolist())]
    out = {"raster": os.path.basename(raster_path), "num_detections": len(recs),
           "crs": (str(crs) if crs is not None else None), "detections": recs}
    with open(out_json, "w") as f:
        _json.dump(out, f, indent=2)
    return out


def gt_to_vectors(raster_path, gt_boxes, out_geojson=None, out_gpkg=None):
    if gt_boxes is None or len(gt_boxes) == 0:
        return None
    with rasterio.open(raster_path) as src:
        T, crs = src.transform, src.crs
    geoms = []
    for (x1, y1, x2, y2) in np.asarray(gt_boxes, np.float32).tolist():
        (X1, Y1) = rasterio.transform.xy(T, y1, x1)   # (row=y, col=x) -> (x_crs, y_crs)
        (X2, Y2) = rasterio.transform.xy(T, y2, x2)
        geoms.append(shp_box(min(X1, X2), min(Y1, Y2), max(X1, X2), max(Y1, Y2)))
    gdf = gpd.GeoDataFrame({"class": "vessel"}, geometry=geoms, crs=crs)
    if out_gpkg:
        gdf.to_file(out_gpkg, driver="GPKG")
        print("GT GPKG", out_gpkg)
    if out_geojson:
        (gdf.to_crs(4326) if crs is not None else gdf).to_file(out_geojson, driver="GeoJSON")
        print("Vector", out_geojson)
    return gdf


def save_gt_json(raster_path, gt_boxes, out_json):
    crs, recs = None, []
    boxes = np.asarray(gt_boxes, np.float32).tolist() if gt_boxes is not None else []
    try:
        with rasterio.open(raster_path) as src:
            T, crs = src.transform, src.crs
            for (x1, y1, x2, y2) in boxes:
                rec = {"bbox_xyxy_pixel": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]}
                if crs is not None:
                    gx, gy = rasterio.transform.xy(T, (y1 + y2) / 2.0, (x1 + x2) / 2.0)
                    rec["center_crs_xy"] = [round(gx, 2), round(gy, 2)]
                recs.append(rec)
    except Exception as e:
        recs = [{"bbox_xyxy_pixel": [round(float(v), 1) for v in b]} for b in boxes]
    out = {"raster": os.path.basename(raster_path), "num_gt": len(recs),
           "crs": (str(crs) if crs is not None else None), "ground_truth": recs}
    with open(out_json, "w") as f:
        _json.dump(out, f, indent=2)
    return out


def _list_layers(gpkg_path):
    # list layer in gpkg
    try:
        import pyogrio
        return [l[0] for l in pyogrio.list_layers(gpkg_path)]
    except Exception:
        pass
    try:
        import fiona
        return list(fiona.listlayers(gpkg_path))
    except Exception:
        return [None]


def inspect_gpkg(gpkg_path):
    import geopandas as gpd
    layers = _list_layers(gpkg_path)
    print("GeoPackage:", gpkg_path)
    print("layers: ", layers)
    for lyr in layers:
        g = gpd.read_file(gpkg_path, layer=lyr) if lyr else gpd.read_file(gpkg_path)
        print(f"\n== layer '{lyr}' == n={len(g)} | CRS={g.crs}")
        print("column:", list(g.columns))
        with_no_geom = [c for c in g.columns if c != g.geometry.name]
        if with_no_geom:
            print(g[with_no_geom].head(3).to_string())
    return layers


def load_gt_boxes_from_gpkg(gpkg_path, raster_path, product_col=None, product_value=None,
                            layer=None, verbose=True):
    gdf = gpd.read_file(gpkg_path, layer=layer) if layer else gpd.read_file(gpkg_path)
    if verbose:
        print(f"GT from gpkg: {len(gdf)} annotation | CRS={gdf.crs} | column={list(gdf.columns)}")
    if product_col and product_value is not None:
        assert product_col in gdf.columns, f"Not found '{product_col}'"
        m = gdf[product_col].astype(str).str.contains(str(product_value), case=False, na=False)
        gdf = gdf[m]
        if verbose:
            print(f"Filtered {product_col} = '{product_value}': {len(gdf)} annotation")
    with rasterio.open(raster_path) as src:
        crs, W, H = src.crs, src.width, src.height
        if crs is not None and gdf.crs is not None and str(gdf.crs) != str(crs):
            gdf = gdf.to_crs(crs)
        boxes = []
        for geom in gdf.geometry:
            if geom is None or geom.is_empty:
                continue
            minx, miny, maxx, maxy = geom.bounds
            r1, c1 = src.index(minx, maxy)    
            r2, c2 = src.index(maxx, miny)    
            x1, x2 = sorted((c1, c2)); y1, y2 = sorted((r1, r2))
            if x2 < 0 or y2 < 0 or x1 > W or y1 > H:
                continue
            boxes.append([max(0, x1), max(0, y1), min(W, x2), min(H, y2)])
    if verbose:
        print(f"-> {len(boxes)} GT in the image.")
    return np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), np.float32)


import re as _re
import glob as _glob


def parse_tile_date(name):
    # 'T34VEM_20220813T0959_TCI.jp2' -> ('34VEM', '20220813')."""
    base = os.path.basename(str(name))
    mt = _re.search(r"T?(\d{2}[A-Za-z]{3})", base)      # ô MGRS: 2 số + 3 chữ (vd 34VEM)
    md = _re.search(r"(?<!\d)(\d{8})(?!\d)", base)       # ngày YYYYMMDD
    return (mt.group(1).upper() if mt else None), (md.group(1) if md else None)


def _resolve_gpkg(gpkg_source, tile):
    if not os.path.isdir(gpkg_source):
        return gpkg_source
    cands = sorted(_glob.glob(os.path.join(gpkg_source, "*.gpkg")))
    if not tile:
        raise FileNotFoundError(f"File list: {[os.path.basename(c) for c in cands]}")
    for c in cands:
        stem = os.path.splitext(os.path.basename(c))[0].upper().lstrip("T")
        if stem == tile.upper().lstrip("T"):
            return c
    raise FileNotFoundError(f"Not found gpkg for '{tile}' in {gpkg_source}. ")


def _resolve_layer(gpkg, date, layer):
    if layer:
        return str(layer)
    lyrs = [str(l) for l in _list_layers(gpkg) if l]
    if date and date in lyrs:
        return date
    if len(lyrs) == 1:
        return lyrs[0]
    raise ValueError(
        f"'{os.path.basename(gpkg)}' has layers : {lyrs}. ")


def gt_boxes_for_raster(raster_path, gpkg_source, layer=None, verbose=True):
    tile, date = parse_tile_date(raster_path)
    gpkg = _resolve_gpkg(gpkg_source, tile)
    lyr = _resolve_layer(gpkg, date, layer)
    if verbose:
        print(f"GT: tile={tile} date={date} -> {os.path.basename(gpkg)} [layer '{lyr}']")
    return load_gt_boxes_from_gpkg(gpkg, raster_path, layer=lyr, verbose=verbose)


def _iou_matrix(A, B):
    if len(A) == 0 or len(B) == 0:
        return np.zeros((len(A), len(B)), np.float32)
    A = A[:, None, :]; B = B[None, :, :]
    ix1 = np.maximum(A[..., 0], B[..., 0]); iy1 = np.maximum(A[..., 1], B[..., 1])
    ix2 = np.minimum(A[..., 2], B[..., 2]); iy2 = np.minimum(A[..., 3], B[..., 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aA = (A[..., 2] - A[..., 0]) * (A[..., 3] - A[..., 1])
    aB = (B[..., 2] - B[..., 0]) * (B[..., 3] - B[..., 1])
    return inter / (aA + aB - inter + 1e-9)


def _match(pred_boxes, pred_scores, gt_boxes, iou_thr):
    # calculate AP
    pb = np.asarray(pred_boxes, np.float32); ps = np.asarray(pred_scores, np.float32)
    gt = np.asarray(gt_boxes, np.float32)
    order = np.argsort(-ps)
    iou = _iou_matrix(pb[order], gt) if len(pb) else np.zeros((0, len(gt)))
    matched = np.zeros(len(gt), bool)
    tp = np.zeros(len(order), bool)
    for i in range(len(order)):
        if len(gt) == 0:
            break
        j = int(np.argmax(np.where(matched, -1.0, iou[i])))
        if iou[i, j] >= iou_thr and not matched[j]:
            tp[i] = True; matched[j] = True
    return ps[order], tp, len(gt)


def _average_precision(tp_desc, n_gt):
    if n_gt == 0:
        return float("nan")
    tp = np.cumsum(tp_desc.astype(np.float64))
    fp = np.cumsum((~tp_desc).astype(np.float64))
    recall = tp / (n_gt + 1e-9)
    precision = tp / np.maximum(tp + fp, 1e-9)
    ap = 0.0
    for r in np.linspace(0, 1, 101):
        p = precision[recall >= r].max() if np.any(recall >= r) else 0.0
        ap += p / 101.0
    return float(ap)


def evaluate_detections(pred_boxes, pred_scores, gt_boxes, conf=0.25, iou_thr=0.5):
    pb = np.asarray(pred_boxes, np.float32); ps = np.asarray(pred_scores, np.float32)
    n_gt = int(len(gt_boxes))

    sc, tp, _ = _match(pb, ps, gt_boxes, iou_thr)
    ap = _average_precision(tp, n_gt)
    maps = [_average_precision(_match(pb, ps, gt_boxes, t)[1], n_gt) for t in np.arange(0.5, 1.0, 0.05)]
    map5095 = float(np.nanmean(maps)) if n_gt else float("nan")

    keep = ps >= conf
    _, tp_c, _ = _match(pb[keep], ps[keep], gt_boxes, iou_thr)
    TP = int(tp_c.sum()); FP = int(len(tp_c) - TP); FN = int(n_gt - TP)
    P = TP / (TP + FP + 1e-9); R = TP / (TP + FN + 1e-9); F1 = 2 * P * R / (P + R + 1e-9)
    return {"conf": conf, "iou_thr": iou_thr, "n_pred": int(len(pb)), "n_gt": n_gt,
            "TP": TP, "FP": FP, "FN": FN,
            "precision": round(P, 4), "recall": round(R, 4), "f1": round(F1, 4),
            f"AP{int(iou_thr * 100)}": round(ap, 4), "mAP50_95": round(map5095, 4)}


print("Loaded helpers.py")

## 4b. Mask mây (Sentinel-2 L1C / L2A)

Mô hình hay nhận nhầm **cụm mây nhỏ, sáng** thành tàu. Ta dùng mask mây của Copernicus để loại các dự đoán rơi vào vùng mây.

- **L2A**: dùng lớp phân loại cảnh **SCL** (8=mây vừa, 9=mây cao, 10=cirrus), hoặc `MSK_CLDPRB` / `MSK_CLASSI`.
- **L1C**: dùng `MSK_CLASSI` (baseline ≥ 04.00) hoặc `MSK_CLOUDS` (GML, baseline cũ).
- **Chỉ có ảnh TCI** (không có band khác): **mặc định bỏ qua lọc mây** cho ảnh đó (đặt `CFG['CLOUD_SKIP_IF_TCI_ONLY']=False` để dùng fallback độ sáng, kém chính xác hơn).

Mức xử lý (L1C/L2A) được nhận từ **tên file**; nếu tên không chuẩn thì đọc **metadata** (`MTD_MSIL*.xml`, `MTD_TL.xml`) hoặc suy từ **cấu trúc thư mục**.

Chế độ (`CFG['CLOUD_MASK_MODE']`): **`after`** (khuyến nghị — infer trên ảnh gốc rồi bỏ box trùng mây) hoặc **`before`** (đen hoá pixel mây trước khi infer).

**Nguồn mask nâng cao** — đặt `CFG['CLOUD_SOURCE'] = 'omnicloudmask'` để dùng **OmniCloudMask** (CNN, chính xác hơn hẳn ở mây mỏng/nhỏ, phát hiện được cả bóng mây). Nó đọc band **B04/B03/B8A** trong folder `.SAFE` (không dùng được nếu chỉ có ảnh TCI) và nên chạy trên **GPU**. Thiếu band hoặc gặp lỗi thì tự quay về nguồn `auto`.

In [ ]:
# =============================================================================
#  Mask mây cho Sentinel-2 (L1C / L2A)
# -----------------------------------------------------------------------------
#  - Nhận dạng mức xử lý sản phẩm (L1C hay L2A) từ TÊN FILE; nếu tên không chuẩn
#    thì đọc METADATA (MTD_MSIL1C.xml / MTD_MSIL2A.xml / MTD_TL.xml) hoặc suy ra
#    từ cấu trúc thư mục (có SCL / R10m,R20m,R60m => L2A).
#  - Tự tìm nguồn mask mây tốt nhất trong folder Copernicus (.SAFE):
#       L2A: SCL  >  MSK_CLDPRB  >  MSK_CLASSI
#       L1C: MSK_CLASSI (baseline >= 04.00)  >  MSK_CLOUDS (GML, baseline cũ)
#       Chỉ có TCI: fallback theo độ sáng/độ bão hoà.
#  - "after"  (khuyến nghị): infer trên ảnh gốc, sau đó BỎ các box trùng vùng mây.
#  - "before" : xoá (đen hoá) pixel mây trước khi infer.
# =============================================================================
import os, re, glob
import numpy as np
import cv2
import rasterio
from rasterio.enums import Resampling
from affine import Affine

# ---- tên file Sentinel-2 -----------------------------------------------------
_S2_BAND_RE = re.compile(r"_(B0[1-9]|B1[0-2]|B8A|AOT|WVP|SCL|PVI|TCI)_?\d*m?\.(jp2|tif|tiff)$", re.I)
_S2_MASK_RE = re.compile(r"MSK_[A-Z0-9]+", re.I)
_TCI_RE     = re.compile(r"_TCI(_\d{2}m)?\.(jp2|tif|tiff)$", re.I)


def is_tci(path):
    return bool(_TCI_RE.search(os.path.basename(path)))


def is_s2_band_or_mask(path):
    """True cho các file band/phụ trợ Sentinel-2 (B01..B12, SCL, MSK_*, PVI...)
    — đây là đầu vào để tính mask, KHÔNG phải ảnh để chạy detect. TCI được loại
    khỏi nhóm này vì TCI chính là ảnh ta infer."""
    if is_tci(path):
        return False
    b = os.path.basename(path)
    return bool(_S2_BAND_RE.search(b) or _S2_MASK_RE.search(b))


# ---- L1C vs L2A --------------------------------------------------------------
def find_safe_root(path):
    """Đi ngược lên tới thư mục gốc sản phẩm: kết thúc bằng .SAFE, hoặc chứa
    MTD_MSIL1C.xml / MTD_MSIL2A.xml, hoặc chứa thư mục GRANULE."""
    d = path if os.path.isdir(path) else os.path.dirname(path)
    prev = None
    while d and d != prev:
        if d.upper().endswith(".SAFE"):
            return d
        try:
            entries = set(os.listdir(d))
        except OSError:
            entries = set()
        if {"MTD_MSIL1C.xml", "MTD_MSIL2A.xml"} & entries or "GRANULE" in entries:
            return d
        prev, d = d, os.path.dirname(d)
    return None


def _level_from_metadata(safe_root):
    cands = [os.path.join(safe_root, "MTD_MSIL1C.xml"),
             os.path.join(safe_root, "MTD_MSIL2A.xml")]
    cands += glob.glob(os.path.join(safe_root, "MTD_MSIL*.xml"))
    cands += glob.glob(os.path.join(safe_root, "GRANULE", "*", "MTD_TL.xml"))
    for xml in cands:
        if not os.path.exists(xml):
            continue
        try:
            up = open(xml, "r", errors="ignore").read(20000).upper()
        except OSError:
            continue
        if "S2MSI2A" in up or "LEVEL-2A" in up:
            return "L2A"
        if "S2MSI1C" in up or "LEVEL-1C" in up:
            return "L1C"
    return None


def _level_from_structure(safe_root):
    if glob.glob(os.path.join(safe_root, "GRANULE", "*", "IMG_DATA", "R*m")):
        return "L2A"
    if glob.glob(os.path.join(safe_root, "GRANULE", "*", "**", "*_SCL_*.jp2"), recursive=True):
        return "L2A"
    if glob.glob(os.path.join(safe_root, "GRANULE", "*", "IMG_DATA", "*_B0*.jp2")):
        return "L1C"
    return None


def detect_product_level(path, safe_root=None, verbose=False):
    """Trả 'L1C' | 'L2A' | None cho 1 raster hoặc folder sản phẩm.
    Thứ tự: (1) token trong tên file, (2) hậu tố _TCI_10m chỉ có ở L2A,
    (3) metadata XML, (4) cấu trúc thư mục. -> Trả lời câu hỏi: tên file không
    chuẩn vẫn xác định được mức nhờ (3)-(4)."""
    up = str(path).upper()
    if "MSIL2A" in up or "MSI_L2A" in up:
        lvl = "L2A"
    elif "MSIL1C" in up or "MSI_L1C" in up:
        lvl = "L1C"
    elif re.search(r"_TCI_\d{2}M", up):
        lvl = "L2A"
    else:
        root = safe_root or find_safe_root(path)
        lvl = (_level_from_metadata(root) or _level_from_structure(root)) if root else None
    if verbose:
        print(f"[cloud] product level for {os.path.basename(str(path))}: {lvl}")
    return lvl


# ---- tìm nguồn mask mây ------------------------------------------------------
def _granule_root(raster_path):
    d = os.path.dirname(raster_path)
    for _ in range(4):
        if os.path.isdir(os.path.join(d, "IMG_DATA")) or os.path.isdir(os.path.join(d, "QI_DATA")):
            return d
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    return None


def _log(kind, path, verbose):
    if verbose:
        print(f"[cloud] source = {kind}: {os.path.basename(path)}")
    return kind, path


def find_cloud_source(raster_path, level=None, verbose=False):
    """(kind, path) cho nguồn mask tốt nhất cạnh TCI.
    L2A: 'scl' > 'cldprb' > 'classi'; L1C: 'classi' > 'clouds_gml';
    không có metadata -> ('brightness', tci)."""
    level = level or detect_product_level(raster_path)
    g = _granule_root(raster_path)

    def first(*patterns):
        for pat in patterns:
            hit = sorted(glob.glob(pat, recursive=True))
            if hit:
                return hit[0]
        return None

    if g:
        img, qi = os.path.join(g, "IMG_DATA"), os.path.join(g, "QI_DATA")
        if level == "L2A":
            scl = first(os.path.join(img, "R20m", "*_SCL_20m.jp2"),
                        os.path.join(img, "R60m", "*_SCL_60m.jp2"),
                        os.path.join(img, "**", "*_SCL_*.jp2"))
            if scl:
                return _log("scl", scl, verbose)
            cldprb = first(os.path.join(qi, "MSK_CLDPRB_20m.jp2"),
                           os.path.join(qi, "MSK_CLDPRB_60m.jp2"),
                           os.path.join(qi, "MSK_CLDPRB*.jp2"))
            if cldprb:
                return _log("cldprb", cldprb, verbose)
        classi = first(os.path.join(qi, "MSK_CLASSI_B00.jp2"),
                       os.path.join(qi, "MSK_CLASSI*.jp2"))
        if classi:
            return _log("classi", classi, verbose)
        gml = first(os.path.join(qi, "MSK_CLOUDS_B00.gml"),
                    os.path.join(qi, "MSK_CLOUDS*.gml"))
        if gml:
            return _log("clouds_gml", gml, verbose)
    return _log("brightness", raster_path, verbose)


# ---- CloudMask: mask boolean trên lưới riêng (array + transform + crs) --------
class CloudMask:
    def __init__(self, mask, transform, crs=None, kind="unknown"):
        self.mask = np.ascontiguousarray(mask, dtype=bool)
        self.transform = transform
        self.crs = crs
        self.kind = kind

    @property
    def coverage(self):
        return float(self.mask.mean()) if self.mask.size else 0.0

    def box_cloud_fraction(self, boxes, tci_transform):
        """Tỉ lệ (0..1) diện tích mỗi box (px của TCI, [x1,y1,x2,y2]) bị mây phủ."""
        boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
        if len(boxes) == 0 or self.mask.size == 0:
            return np.zeros(len(boxes), np.float32)
        inv = ~self.transform
        Hm, Wm = self.mask.shape
        out = np.zeros(len(boxes), np.float32)
        for i, (x1, y1, x2, y2) in enumerate(boxes):
            (X1, Y1) = tci_transform * (x1, y1)
            (X2, Y2) = tci_transform * (x2, y2)
            (c1, r1) = inv * (X1, Y1)
            (c2, r2) = inv * (X2, Y2)
            cmin, cmax = sorted((c1, c2)); rmin, rmax = sorted((r1, r2))
            cs = max(0, int(np.floor(cmin))); ce = min(Wm, int(np.ceil(cmax)))
            rs = max(0, int(np.floor(rmin))); re_ = min(Hm, int(np.ceil(rmax)))
            if ce <= cs or re_ <= rs:
                cc = min(Wm - 1, max(0, int((cmin + cmax) / 2)))
                rr = min(Hm - 1, max(0, int((rmin + rmax) / 2)))
                out[i] = float(self.mask[rr, cc]); continue
            out[i] = float(self.mask[rs:re_, cs:ce].mean())
        return out

    def window_mask(self, tci_transform, ox, oy, tw, th):
        """Mask mây (th, tw) cho 1 cửa sổ TCI tại offset (ox, oy) — dùng cho 'before'."""
        inv = ~self.transform
        Hm, Wm = self.mask.shape
        (X0, Y0) = tci_transform * (ox, oy)
        (X1, Y1) = tci_transform * (ox + tw, oy + th)
        (c0, r0) = inv * (X0, Y0); (c1, r1) = inv * (X1, Y1)
        cs = max(0, int(np.floor(min(c0, c1)))); ce = min(Wm, int(np.ceil(max(c0, c1))))
        rs = max(0, int(np.floor(min(r0, r1)))); re_ = min(Hm, int(np.ceil(max(r0, r1))))
        if ce <= cs or re_ <= rs:
            return np.zeros((th, tw), bool)
        sub = self.mask[rs:re_, cs:ce].astype(np.uint8)
        return cv2.resize(sub, (tw, th), interpolation=cv2.INTER_NEAREST).astype(bool)


def _dilate(mask, px):
    if px <= 0:
        return mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * px + 1, 2 * px + 1))
    return cv2.dilate(mask.astype(np.uint8), k).astype(bool)


def _brightness_mask(tci_path, bright_thr, sat_thr, max_side=2048):
    with rasterio.open(tci_path) as src:
        W, H = src.width, src.height
        scale = min(1.0, max_side / max(W, H))
        ow, oh = max(1, int(W * scale)), max(1, int(H * scale))
        idx = [1, 2, 3] if src.count >= 3 else [1, 1, 1]
        arr = src.read(indexes=idx, out_shape=(3, oh, ow)).astype(np.float32)
        transform = src.transform * Affine.scale(W / ow, H / oh)
        crs = src.crs
    mx = arr.max() or 1.0
    rgb = arr / mx
    mn = rgb.min(axis=0); mxc = rgb.max(axis=0)
    bright = mxc >= bright_thr
    sat = (mxc - mn) / (mxc + 1e-6)
    return (bright & (sat <= sat_thr)), transform, crs


def _rasterize_gml(gml_path, raster_path, max_side=4096):
    import geopandas as gpd
    from rasterio.features import rasterize
    with rasterio.open(raster_path) as src:
        W, H, crs, T = src.width, src.height, src.crs, src.transform
    scale = min(1.0, max_side / max(W, H))
    ow, oh = max(1, int(W * scale)), max(1, int(H * scale))
    transform = T * Affine.scale(W / ow, H / oh)
    try:
        gdf = gpd.read_file(gml_path)
        if crs is not None and gdf.crs is not None and str(gdf.crs) != str(crs):
            gdf = gdf.to_crs(crs)
        geoms = [g for g in gdf.geometry if g is not None and not g.is_empty]
    except Exception:
        geoms = []
    if not geoms:
        return np.zeros((oh, ow), bool), transform, crs
    m = rasterize([(g, 1) for g in geoms], out_shape=(oh, ow),
                  transform=transform, fill=0, dtype="uint8")
    return m.astype(bool), transform, crs


# ---- OmniCloudMask (deep learning; cần band B04/B03/B8A trong .SAFE) ----------
def _find_s2_band(granule, band, level):
    """Tìm file 1 band Sentinel-2 (vd 'B04') trong 1 granule .SAFE."""
    img = os.path.join(granule, "IMG_DATA")
    if level == "L2A":
        for res in ("R10m", "R20m", "R60m"):
            hit = sorted(glob.glob(os.path.join(img, res, f"*_{band}_*m.jp2")))
            if hit:
                return hit[0]
        hit = sorted(glob.glob(os.path.join(img, "**", f"*_{band}_*.jp2"), recursive=True))
        return hit[0] if hit else None
    hit = sorted(glob.glob(os.path.join(img, f"*_{band}.jp2")))
    if not hit:
        hit = sorted(glob.glob(os.path.join(img, "**", f"*_{band}.jp2"), recursive=True))
    return hit[0] if hit else None


def _omnicloudmask_mask(raster_path, cfg, verbose=True):
    """Mask mây bằng OmniCloudMask (CNN). Đọc Red=B04, Green=B03, NIR=B8A ở dạng
    raw DN (đúng như loader load_s2 của thư viện), suy luận, rồi lấy các lớp mây
    (1=dày, 2=mỏng) và tuỳ chọn bóng mây (3). Trả (mask, transform, crs)."""
    from omnicloudmask import predict_from_array
    level = detect_product_level(raster_path)
    g = _granule_root(raster_path)
    if g is None:
        raise RuntimeError("cần folder .SAFE để đọc band B04/B03/B8A, không chỉ ảnh TCI")
    red = _find_s2_band(g, "B04", level)
    green = _find_s2_band(g, "B03", level)
    nir = _find_s2_band(g, "B8A", level) or _find_s2_band(g, "B08", level)
    if not (red and green and nir):
        raise RuntimeError(f"thiếu band (B04={bool(red)} B03={bool(green)} NIR={bool(nir)})")
    target_res = float(cfg.get("CLOUD_OCM_RES", 20.0))
    with rasterio.open(red) as src:
        natW, natH = src.width, src.height
        nat_res = abs(src.transform.a) or 10.0
        crs, base_T = src.crs, src.transform
    factor = max(1.0, target_res / nat_res)
    outW = max(1, int(round(natW / factor)))
    outH = max(1, int(round(natH / factor)))
    transform = base_T * Affine.scale(natW / outW, natH / outH)

    def _rd(p):
        with rasterio.open(p) as s:
            return s.read(1, out_shape=(outH, outW), resampling=Resampling.bilinear).astype(np.float32)

    arr = np.stack([_rd(red), _rd(green), _rd(nir)], 0)   # (3,H,W) Red,Green,NIR raw DN
    dev = cfg.get("CLOUD_OCM_DEVICE")
    if not dev:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
    if verbose:
        print(f"[cloud] omnicloudmask {outW}x{outH}px @~{target_res:.0f}m device={dev} "
              f"(R={os.path.basename(red)}, NIR={os.path.basename(nir)})")
    pred = predict_from_array(arr, inference_device=dev, mosaic_device=dev,
                              batch_size=int(cfg.get("CLOUD_OCM_BATCH", 1)))
    labels = pred[0] if getattr(pred, "ndim", 2) == 3 else pred
    classes = set(cfg.get("CLOUD_OCM_CLASSES", (1, 2)))   # 1=mây dày, 2=mây mỏng
    if cfg.get("CLOUD_INCLUDE_SHADOW"):
        classes |= {3}                                    # 3=bóng mây
    mask = np.isin(labels, list(classes))
    return mask, transform, crs


def build_cloud_mask(raster_path, cfg=None, kind=None, source_path=None, verbose=True):
    """Dựng CloudMask cho raster_path; tự tìm nguồn nếu chưa cho. cfg=CFG để lấy ngưỡng."""
    cfg = cfg or {}
    if kind is None and str(cfg.get("CLOUD_SOURCE", "auto")).lower() == "omnicloudmask":
        try:
            mask, transform, crs = _omnicloudmask_mask(raster_path, cfg, verbose=verbose)
            mask = _dilate(mask, cfg.get("CLOUD_DILATE_PX", 0))
            cm = CloudMask(mask, transform, crs, kind="omnicloudmask")
            if verbose:
                print(f"[cloud] mask kind=omnicloudmask shape={cm.mask.shape} "
                      f"coverage={cm.coverage*100:.1f}%")
            return cm
        except Exception as e:
            print("[cloud] omnicloudmask lỗi -> quay về nguồn auto:", str(e).splitlines()[0])
    if kind is None:
        kind, source_path = find_cloud_source(raster_path, verbose=verbose)
    source_path = source_path or raster_path

    with rasterio.open(source_path) as src:
        transform, crs = src.transform, src.crs
        if kind == "scl":
            scl = src.read(1)
            classes = set(cfg.get("CLOUD_SCL_CLASSES", (8, 9, 10)))
            if cfg.get("CLOUD_INCLUDE_SHADOW"): classes |= {3}
            if cfg.get("CLOUD_INCLUDE_SNOW"):   classes |= {11}
            mask = np.isin(scl, list(classes))
        elif kind == "cldprb":
            mask = src.read(1).astype(np.float32) >= cfg.get("CLOUD_CLDPRB_THR", 40)
        elif kind == "classi":
            a = src.read()
            bands = a[:2] if a.shape[0] >= 2 else a
            mask = (bands > 0).any(axis=0)
        elif kind == "clouds_gml":
            mask, transform, crs = _rasterize_gml(source_path, raster_path)
        else:
            mask, transform, crs = _brightness_mask(
                source_path, cfg.get("CLOUD_BRIGHT_THR", 0.60), cfg.get("CLOUD_SAT_THR", 0.20))

    mask = _dilate(mask, cfg.get("CLOUD_DILATE_PX", 0))
    cm = CloudMask(mask, transform, crs, kind=kind)
    if verbose:
        print(f"[cloud] mask kind={kind} shape={cm.mask.shape} coverage={cm.coverage*100:.1f}%")
    return cm


def filter_boxes_by_cloud(boxes, scores, cloud_mask, tci_transform, frac_thr=0.35,
                          mode="fraction", protect_conf=0.0):
    """Tách detections thành giữ / bỏ theo mức trùng mây.
    mode='fraction': bỏ nếu >= frac_thr diện tích box là mây; 'center': bỏ nếu tâm box là mây.
    protect_conf>0: KHÔNG loại các dự đoán có conf >= protect_conf (bảo vệ tàu thật,
    tránh mất tàu khi mask mây quá mạnh tay / gán nhầm thân tàu là mây)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    if len(boxes) == 0:
        z = np.zeros((0, 4), np.float32)
        return z, scores[:0], z, scores[:0]
    if mode == "center":
        cx = (boxes[:, 0] + boxes[:, 2]) / 2; cy = (boxes[:, 1] + boxes[:, 3]) / 2
        frac = cloud_mask.box_cloud_fraction(np.stack([cx, cy, cx, cy], 1), tci_transform)
    else:
        frac = cloud_mask.box_cloud_fraction(boxes, tci_transform)
    drop = frac >= frac_thr
    if protect_conf and protect_conf > 0:
        drop = drop & (scores < float(protect_conf))
    return boxes[~drop], scores[~drop], boxes[drop], scores[drop]


# ---- Lọc cirrus bằng band B10 (1375nm) — CHỈ L1C (L2A đã bỏ B10) -------------
def _box_mean_values(values, val_transform, boxes, tci_transform):
    """Giá trị trung bình của 'values' (lưới riêng) dưới mỗi box (px của TCI)."""
    boxes = np.asarray(boxes, np.float64).reshape(-1, 4)
    out = np.zeros(len(boxes), np.float32)
    if len(boxes) == 0 or values.size == 0:
        return out
    inv = ~val_transform
    H, W = values.shape
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        (X1, Y1) = tci_transform * (x1, y1)
        (X2, Y2) = tci_transform * (x2, y2)
        (c1, r1) = inv * (X1, Y1)
        (c2, r2) = inv * (X2, Y2)
        cs = max(0, int(np.floor(min(c1, c2)))); ce = min(W, int(np.ceil(max(c1, c2))))
        rs = max(0, int(np.floor(min(r1, r2)))); re_ = min(H, int(np.ceil(max(r1, r2))))
        if ce <= cs or re_ <= rs:
            cc = min(W - 1, max(0, int((c1 + c2) / 2)))
            rr = min(H - 1, max(0, int((r1 + r2) / 2)))
            out[i] = values[rr, cc]
        else:
            out[i] = values[rs:re_, cs:ce].mean()
    return out


def filter_boxes_by_b10_cirrus(raster_path, boxes, scores, cfg, verbose=True):
    """Bỏ các box là CIRRUS/mây dựa trên band B10 (1375nm) của L1C.
    Cơ sở vật lý: hơi nước tầng thấp hấp thụ 1375nm nên mặt biển & tàu gần như TỐI
    ở B10, còn mây/cirrus (trên cao) thì SÁNG. -> lọc theo B10 an toàn cho tàu và
    KHÔNG phụ thuộc độ tin cậy (bắt được cả FP mây có conf cao mà PROTECT_CONF che).
    Ngưỡng theo CHÊNH LỆCH reflectance so với nền, nên offset baseline tự triệt tiêu.
    Tự bỏ qua với L2A (không có B10) hoặc khi thiếu band. Trả (keep_b, keep_s, drop_b, drop_s)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    z, zs = np.zeros((0, 4), np.float32), np.zeros((0,), np.float32)
    if not cfg.get("CLOUD_B10", True) or len(boxes) == 0:
        return boxes, scores, z, zs
    if detect_product_level(raster_path) == "L2A":
        return boxes, scores, z, zs                        # L2A không có B10
    g = _granule_root(raster_path)
    b10 = _find_s2_band(g, "B10", detect_product_level(raster_path)) if g else None
    if not b10:
        if verbose:
            print("[cloud] không có band B10 -> bỏ qua lọc cirrus")
        return boxes, scores, z, zs
    with rasterio.open(b10) as src:
        arr = src.read(1).astype(np.float32); b10_T = src.transform
    with rasterio.open(raster_path) as src:
        tci_T = src.transform
    valid = arr[arr > 0]
    bg = float(np.percentile(valid, cfg.get("CLOUD_B10_BG_PCT", 20))) if valid.size else 0.0
    excess = (_box_mean_values(arr, b10_T, boxes, tci_T) - bg) / 10000.0
    drop = excess >= float(cfg.get("CLOUD_B10_THR", 0.010))
    if verbose and bool(drop.any()):
        print(f"[cloud] B10 cirrus: bỏ {int(drop.sum())}/{len(boxes)} box "
              f"(chênh reflectance >= {cfg.get('CLOUD_B10_THR', 0.010)})")
    return boxes[~drop], scores[~drop], boxes[drop], scores[drop]


def _has_cloud_source(raster_path, cfg):
    """True nếu có nguồn mask THẬT kèm theo ảnh (SCL/CLDPRB/CLASSI/GML, hoặc đủ
    band cho OmniCloudMask). False = 'chỉ có ảnh TCI', không có gì để mask."""
    if str(cfg.get("CLOUD_SOURCE", "auto")).lower() == "omnicloudmask":
        g = _granule_root(raster_path)
        if g is not None:
            lvl = detect_product_level(raster_path)
            if (_find_s2_band(g, "B04", lvl) and _find_s2_band(g, "B03", lvl)
                    and (_find_s2_band(g, "B8A", lvl) or _find_s2_band(g, "B08", lvl))):
                return True
    kind, _ = find_cloud_source(raster_path, verbose=False)
    return kind != "brightness"


def get_cloud_mask_for(raster_path, cfg, verbose=True):
    """Trả CloudMask cho raster nếu bật CLOUD_MASK và mode != 'off', ngược lại None.
    Bọc lỗi để 1 ảnh hỏng mask không làm chết cả vòng lặp."""
    if not cfg.get("CLOUD_MASK", False) or str(cfg.get("CLOUD_MASK_MODE", "after")).lower() == "off":
        return None
    if cfg.get("CLOUD_SKIP_IF_TCI_ONLY", True) and not _has_cloud_source(raster_path, cfg):
        if verbose:
            print("[cloud] chỉ có ảnh TCI (không có band/mask kèm theo) -> bỏ qua lọc mây cho ảnh này")
        return None
    try:
        lvl = detect_product_level(raster_path, verbose=verbose)
        return build_cloud_mask(raster_path, cfg=cfg, verbose=verbose)
    except Exception as e:
        print("[cloud] không dựng được mask, bỏ qua lọc mây:", str(e).splitlines()[0])
        return None


def apply_cloud_filter(raster_path, boxes, scores, cloud_mask, cfg):
    """Lọc detections sau infer (mode 'after'). Trả (boxes, scores, n_dropped)."""
    if cloud_mask is None or str(cfg.get("CLOUD_MASK_MODE", "after")).lower() != "after":
        return boxes, scores, 0
    with rasterio.open(raster_path) as src:
        T = src.transform
    kb, ks, db, ds = filter_boxes_by_cloud(
        boxes, scores, cloud_mask, T,
        frac_thr=cfg.get("CLOUD_FRAC_THR", 0.35),
        mode=cfg.get("CLOUD_FILTER", "fraction"),
        protect_conf=cfg.get("CLOUD_PROTECT_CONF", 0.5))
    return kb, ks, int(len(db))


print("Loaded cloud-mask helpers.")


## 5. Download image from Drive

In [ ]:
import gdown, glob, zipfile

RASTER_EXTS = (".jp2", ".tif", ".tiff", ".png", ".jpg", ".jpeg")


def extract_zips(root, delete=False):
    """Giải nén mọi *.zip trong thư mục tải về. Mỗi zip tải từ Copernicus đã chứa
    sẵn cấu trúc đầy đủ của 1 ảnh Sentinel-2 (folder .SAFE với GRANULE/IMG_DATA/
    QI_DATA + MTD_*.xml). Idempotent: bỏ qua nếu đã giải nén từ lần chạy trước.
    Trả về số file zip đã xử lý."""
    zips = sorted(glob.glob(os.path.join(root, "**", "*.zip"), recursive=True))
    for z in zips:
        dest = os.path.dirname(z)
        try:
            with zipfile.ZipFile(z) as zf:
                tops = sorted({n.split("/", 1)[0] for n in zf.namelist() if n.strip("/")})
                if tops and all(os.path.exists(os.path.join(dest, t)) for t in tops):
                    print(f"  [zip] đã giải nén trước đó, bỏ qua: {os.path.basename(z)}")
                else:
                    print(f"  [zip] giải nén {os.path.basename(z)} -> {tops[:1]} ...")
                    zf.extractall(dest)
            if delete:
                os.remove(z)
                print(f"  [zip] đã xoá {os.path.basename(z)} để tiết kiệm dung lượng")
        except zipfile.BadZipFile:
            print(f"  [zip] file hỏng, bỏ qua: {os.path.basename(z)}")
        except Exception as e:
            print(f"  [zip] lỗi {os.path.basename(z)}: {str(e).splitlines()[0]}")
    return len(zips)


def select_inference_rasters(root):
    """Chọn các ảnh để CHẠY DETECT trong 1 cây thư mục (sau khi đã giải nén .zip).
    Đầu vào có thể là folder Copernicus (.SAFE) đầy đủ, hoặc vài ảnh TCI rời:
      - Giữ file TCI (ảnh true-color để infer).
      - Bỏ file band/phụ trợ Sentinel-2 (B01..B12, SCL, MSK_*, PVI...) — chúng
        chỉ dùng để tính mask mây, không phải ảnh để detect.
      - Giữ ảnh rời khác (vd .tif/.png không thuộc SAFE).
    Trả (rasters, safe_root_map) với safe_root_map[raster] = thư mục gốc .SAFE (hoặc None)."""
    all_files = [p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
                 if p.lower().endswith(RASTER_EXTS)]
    has_tci = any(is_tci(p) for p in all_files)
    rasters = []
    for p in all_files:
        if is_tci(p):
            rasters.append(p)
        elif is_s2_band_or_mask(p):
            continue                       # band / mask -> không detect
        elif not has_tci:
            rasters.append(p)              # ảnh rời (không có TCI trong bộ)
    rasters = sorted(set(rasters))
    safe_root_map = {p: find_safe_root(p) for p in rasters}
    return rasters, safe_root_map


# ---- chạy ----
gdown.download_folder(CFG["DRIVE_FOLDER_URL"], output=CFG["DL_DIR"], quiet=False, use_cookies=False)

n_zip = extract_zips(CFG["DL_DIR"], delete=CFG.get("UNZIP_DELETE_ZIP", False))
if n_zip:
    print(f"Giải nén xong {n_zip} file .zip.")

raster_files, SAFE_ROOTS = select_inference_rasters(CFG["DL_DIR"])
if CFG["LIMIT"]:
    raster_files = raster_files[:CFG["LIMIT"]]

print(f"\nDownloaded. There are {len(raster_files)} images to infer")
for p in raster_files:
    lvl = detect_product_level(p, safe_root=SAFE_ROOTS.get(p)) or "?"
    tag = "SAFE" if SAFE_ROOTS.get(p) else "loose"
    print(f"  - [{lvl:>3}/{tag}] {os.path.relpath(p, CFG['DL_DIR'])}  ({os.path.getsize(p) / 1e6:.1f} MB)")
assert raster_files, "Not found any image to infer (TCI hoặc ảnh rời). Kiểm tra lại zip/thư mục Drive."

if CFG["DRAW_GT"]:
    if os.path.isdir(CFG["GT_GPKG_PATH"]):
        gpkgs = sorted(glob.glob(os.path.join(CFG["GT_GPKG_PATH"], "*.gpkg")))
        print(f"\nGeoPackage label ({len(gpkgs)} tile):", [os.path.basename(g) for g in gpkgs])
        if gpkgs:
            inspect_gpkg(gpkgs[0])
    else:
        inspect_gpkg(CFG["GT_GPKG_PATH"])


## 5b. Infer and export

In [ ]:
import pandas as pd
import shutil
from IPython.display import display


def export_detection_crops(raster_path, boxes, scores, out_dir, pad=40, draw_box=True):
    """Lưu TỪNG phát hiện thành 1 ảnh crop riêng (sắp theo conf giảm dần) vào out_dir.
    Tên file: {stem}_{rank:03d}_conf{score}.png. Trả về số crop đã lưu."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    scores = np.asarray(scores, np.float32).reshape(-1)
    if len(boxes) == 0:
        return 0
    os.makedirs(out_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(raster_path))[0]
    n = 0
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        for rank, i in enumerate(np.argsort(-scores)):
            x1, y1, x2, y2 = (float(v) for v in boxes[i])
            cx0, cy0 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            cw = min(W - cx0, int(x2 - x1) + 2 * pad)
            ch = min(H - cy0, int(y2 - y1) + 2 * pad)
            if cw <= 0 or ch <= 0:
                continue
            crop = _read_rgb_window(src, Window(cx0, cy0, cw, ch)).copy()
            if draw_box:
                cv2.rectangle(crop, (int(x1 - cx0), int(y1 - cy0)),
                              (int(x2 - cx0), int(y2 - cy0)), (255, 0, 0), 1)
            Image.fromarray(crop).save(
                os.path.join(out_dir, f"{stem}_{rank:03d}_conf{float(scores[i]):.2f}.png"))
            n += 1
    return n


def export_hard_negatives(raster_path, neg_boxes, out_dir, tile=320):
    """Cắt 1 tile quanh mỗi box (nghi là mây) thành mẫu NEGATIVE cho YOLO:
    images/{name}.png + labels/{name}.txt (RỖNG = ảnh nền, không có tàu).
    Nạp các tile này vào tập train giúp model tự học 'đây là mây, đừng bắn'.
    Trả về số tile đã lưu."""
    neg_boxes = np.asarray(neg_boxes, np.float32).reshape(-1, 4)
    if len(neg_boxes) == 0:
        return 0
    img_dir = os.path.join(out_dir, "images"); lbl_dir = os.path.join(out_dir, "labels")
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(raster_path))[0]
    n = 0
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        tw, th = min(tile, W), min(tile, H)
        for j, (x1, y1, x2, y2) in enumerate(neg_boxes):
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
            ox = min(max(0, int(round(cx - tw / 2))), max(0, W - tw))
            oy = min(max(0, int(round(cy - th / 2))), max(0, H - th))
            crop = _read_rgb_window(src, Window(ox, oy, tw, th))
            name = f"{stem}_neg{j:03d}"
            Image.fromarray(crop).save(os.path.join(img_dir, name + ".png"))
            open(os.path.join(lbl_dir, name + ".txt"), "w").close()   # nhãn rỗng = background
            n += 1
    return n


def _effective_mode():
    if not CFG.get("DRAW_GT", False):
        return "pred"
    m = str(CFG.get("EXPORT_MODE", "both")).lower()
    if m not in ("pred", "gt", "both"):
        print("Export mode not found")
        return "both"
    return m


def _gt_for(path):
    try:
        gt = gt_boxes_for_raster(path, CFG["GT_GPKG_PATH"], layer=CFG["GT_LAYER"], verbose=False)
    except FileNotFoundError:
        print("Not found ground truth, prediction only")
        return None
    except Exception as e:
        print("Ground truth error, skipping", str(e).splitlines()[0], ")")
        return None
    return gt


MODE = _effective_mode()
CLOUD_ON = CFG.get("CLOUD_MASK", False) and str(CFG.get("CLOUD_MASK_MODE", "after")).lower() != "off"

# --- crops -> zip: dọn thư mục crops trước mỗi lần chạy để zip chỉ chứa kết quả lần này ---
CROPS_DIR = os.path.join(CFG["OUT_DIR"], "crops")
if CFG.get("SAVE_CROPS_ZIP", False):
    shutil.rmtree(CROPS_DIR, ignore_errors=True)
    os.makedirs(CROPS_DIR, exist_ok=True)

# --- hard-negative: gom tile mây của cả run vào 1 dataset (images/ + labels/) ---
HARDNEG_DIR = os.path.join(CFG["OUT_DIR"], "hard_negatives")
if CFG.get("SAVE_HARD_NEG", False):
    shutil.rmtree(HARDNEG_DIR, ignore_errors=True)
    os.makedirs(HARDNEG_DIR, exist_ok=True)

summary = []
for p in raster_files:
    stem = os.path.splitext(os.path.basename(p))[0]
    print("\n" + "=" * 70 + f"\n>>> {os.path.basename(p)}")

    # --- mask mây: dựng 1 lần cho ảnh này ---
    cloud_mask = get_cloud_mask_for(p, CFG, verbose=True) if CLOUD_ON else None
    # 'before' -> xoá pixel mây ngay khi cắt tile; 'after' -> lọc sau
    cm_before = cloud_mask if (cloud_mask is not None and
                               str(CFG.get("CLOUD_MASK_MODE", "after")).lower() == "before") else None

    try:
        b, s, (W, H) = infer_large_raster(
            p, model, conf=CFG["CONF"], tile=CFG["TILE"], overlap=CFG["OVERLAP"],
            imgsz=CFG["IMGSZ"], iou_nms=CFG["IOU_NMS"], nms_iou=CFG["NMS_IOU"],
            batch=CFG["BATCH"], skip_dark=CFG["SKIP_DARK"], device=DEVICE_EFF,
            cloud_mask=cm_before)
    except Exception as e:
        print("Inference error:", str(e).splitlines()[0])
        summary.append({"file": os.path.basename(p), "num_ships": None, "num_gt": None,
                        "cloud_dropped": None, "labeled": None, "mode": None, "size_MB": None})
        continue

    # --- 'after': bỏ các dự đoán rơi vào vùng mây ---
    n_cloud = 0
    drop_boxes = None
    dropped_all = np.zeros((0, 4), np.float32)   # mọi box bị loại là mây (dùng cho hard-negative)
    if cloud_mask is not None and str(CFG.get("CLOUD_MASK_MODE", "after")).lower() == "after":
        with rasterio.open(p) as _src:
            _T = _src.transform
        b, s, db, ds = filter_boxes_by_cloud(
            b, s, cloud_mask, _T,
            frac_thr=CFG.get("CLOUD_FRAC_THR", 0.35),
            mode=CFG.get("CLOUD_FILTER", "fraction"),
            protect_conf=CFG.get("CLOUD_PROTECT_CONF", 0.5))
        n_cloud = int(len(db))
        dropped_all = np.concatenate([dropped_all, np.asarray(db, np.float32).reshape(-1, 4)], 0)
        drop_boxes = db if CFG.get("DRAW_CLOUD", True) else None
        if n_cloud:
            print(f"[cloud] bỏ {n_cloud} dự đoán trùng vùng mây -> còn {len(b)}")

    # --- lọc cirrus bằng B10 (chỉ L1C; an toàn cho tàu, KHÔNG phụ thuộc conf) ---
    if CFG.get("CLOUD_B10", True) and len(b):
        b, s, db2, ds2 = filter_boxes_by_b10_cirrus(p, b, s, CFG, verbose=True)
        if len(db2):
            n_cloud += int(len(db2))
            dropped_all = np.concatenate([dropped_all, np.asarray(db2, np.float32).reshape(-1, 4)], 0)
            if CFG.get("DRAW_CLOUD", True):
                drop_boxes = db2 if drop_boxes is None else np.concatenate([np.asarray(drop_boxes), db2], 0)
            print(f"[cloud] sau lọc B10 còn {len(b)} dự đoán")

    # ground truth if needed
    gt = _gt_for(p) if MODE in ("gt", "both") else None
    has_gt = gt is not None and len(gt) > 0
    file_mode = MODE if has_gt else "pred"
    draw_pred = file_mode in ("pred", "both")
    draw_gt = file_mode in ("gt", "both")

    pred_out = b if draw_pred else None
    gt_out = gt if draw_gt else None

    if draw_pred:
        save_detections_json(p, b, s, os.path.join(CFG["OUT_DIR"], f"{stem}_detections.json"))
    if draw_gt:
        save_gt_json(p, gt, os.path.join(CFG["OUT_DIR"], f"{stem}_gt.json"))

    if CFG["SAVE_VECTOR"]:
        if draw_pred:
            detections_to_vectors(p, b, s,
                                  out_geojson=os.path.join(CFG["OUT_DIR"], f"{stem}_pred.geojson"),
                                  out_gpkg=os.path.join(CFG["OUT_DIR"], f"{stem}_pred.gpkg"))
        if draw_gt:
            gt_to_vectors(p, gt,
                          out_geojson=os.path.join(CFG["OUT_DIR"], f"{stem}_gt.geojson"),
                          out_gpkg=os.path.join(CFG["OUT_DIR"], f"{stem}_gt.gpkg"))

    if CFG["SAVE_GEOTIFF"]:
        try:
            burn_boxes_to_raster(p, os.path.join(CFG["OUT_DIR"], f"{stem}_annotated.tif"),
                                 pred_out, s, gt_boxes=gt_out, driver="GTiff",
                                 thickness=CFG["LINE_THICKNESS"], max_megapixels=CFG["BURN_MAX_MP"])
        except Exception as e:
            print("GeoTIFF err:", str(e).splitlines()[0])
    if CFG["SAVE_JP2"]:
        try:
            burn_boxes_to_raster(p, os.path.join(CFG["OUT_DIR"], f"{stem}_annotated.jp2"),
                                 pred_out, s, gt_boxes=gt_out, driver="JP2OpenJPEG",
                                 thickness=CFG["LINE_THICKNESS"], jp2_quality=100,
                                 max_megapixels=CFG["BURN_MAX_MP"])
        except Exception as e:
            print("JP2 lỗi:", str(e).splitlines()[0])

    _parts = []
    if draw_pred:
        _parts.append(f"{len(b)} predictions (conf≥{CFG['CONF']})")
    if draw_gt:
        _parts.append(f"{len(gt)} GT")
    if n_cloud:
        _parts.append(f"{n_cloud} bỏ do mây")
    draw_overview(p, boxes=pred_out, gt_boxes=gt_out, drop_boxes=drop_boxes,
                  cloud_mask=(cloud_mask if CFG.get("DRAW_CLOUD", True) else None),
                  max_side=CFG["OVERVIEW_MAX"],
                  title=(f"{stem} — " + " · ".join(_parts)) if _parts else stem,
                  save_path=os.path.join(CFG["OUT_DIR"], f"{stem}_overview.png"))
    if draw_pred and len(b):
        show_detection_crops(p, b, s, topk=8,
                             save_path=os.path.join(CFG["OUT_DIR"], f"{stem}_crops.png"))
        # mỗi ảnh -> 1 file zip crop riêng: {stem}_crops.zip
        if CFG.get("SAVE_CROPS_ZIP", False):
            crop_sub = os.path.join(CROPS_DIR, stem)
            shutil.rmtree(crop_sub, ignore_errors=True)
            os.makedirs(crop_sub, exist_ok=True)
            n_saved = export_detection_crops(p, b, s, crop_sub, pad=CFG.get("CROPS_PAD", 40),
                                             draw_box=CFG.get("CROPS_DRAW_BOX", True))
            if n_saved:
                shutil.make_archive(os.path.join(CFG["OUT_DIR"], f"{stem}_crops"), "zip", crop_sub)
                print(f"[crops] {n_saved} crop -> {stem}_crops.zip")

    # --- hard-negative: chọn nguồn box mây rồi cắt tile negative ---
    if CFG.get("SAVE_HARD_NEG", False):
        _sel = str(CFG.get("HARD_NEG_SOURCE", "dropped")).lower()
        if _sel == "kept":
            neg = np.asarray(b, np.float32).reshape(-1, 4)
        elif _sel == "all":
            neg = np.concatenate([np.asarray(b, np.float32).reshape(-1, 4), dropped_all], 0)
        else:  # "dropped"
            neg = dropped_all
        n_neg = export_hard_negatives(p, neg, HARDNEG_DIR, tile=CFG.get("HARD_NEG_TILE", 320))
        if n_neg:
            print(f"[hard-neg] {n_neg} tile ({_sel}) -> hard_negatives/")

    summary.append({"file": os.path.basename(p), "num_ships": int(len(b)),
                    "num_gt": (int(len(gt)) if has_gt else None),
                    "cloud_dropped": (n_cloud if (cloud_mask is not None or CFG.get("CLOUD_B10", True)) else None),
                    "labeled": bool(has_gt), "mode": file_mode,
                    "size_MB": round(os.path.getsize(p) / 1e6, 1)})

if CFG.get("SAVE_CROPS_ZIP", False):
    zips = [f for f in os.listdir(CFG["OUT_DIR"]) if f.endswith("_crops.zip")]
    print(f"\nĐã tạo {len(zips)} file zip crop (mỗi ảnh 1 file) tại {CFG['OUT_DIR']}")

if CFG.get("SAVE_HARD_NEG", False):
    _imgs = os.path.join(HARDNEG_DIR, "images")
    if os.path.isdir(_imgs) and any(os.scandir(_imgs)):
        n = len([f for f in os.listdir(_imgs) if f.endswith(".png")])
        shutil.make_archive(os.path.join(CFG["OUT_DIR"], "hard_negatives"), "zip", HARDNEG_DIR)
        print(f"\nHard-negative: {n} tile -> hard_negatives.zip "
              f"(giải nén ra images/ + labels/ rỗng để trộn vào tập train)")
    else:
        print("\nKhông có hard-negative để gói.")

print("\nSUMMARY")
df = pd.DataFrame(summary)
display(df)
df.to_csv(os.path.join(CFG["OUT_DIR"], "summary.csv"), index=False)
print("Result saved at:", CFG["OUT_DIR"])
